In [ ]:
from bs4 import BeautifulSoup
import requests
from urllib.parse import urljoin
import json, os, time, re, sys
from random import uniform
import requests



# Cambio por url de pagina 1 para despues ir cambiando a cada pagina
BASE = "https://books.toscrape.com/"
url = urljoin(BASE, "catalogue/page-1.html")
next_page = url



rating_number = {"One":1,"Two":2,"Three":3,"Four":4,"Five":5}
items = []


OPENLIB_SEARCH = "https://openlibrary.org/search.json"
GOOGLE_BOOKS = "https://www.googleapis.com/books/v1/volumes"

# funcion para obtener autores usando Open Library y Google Books
def get_authors(title, category, lang="en", timeout=2, max_authors=5):
    """
    Devuelve una LISTA de autores usando Open Library y como fallback Google Books.
    Siempre retorna al menos ["Desconocido"] si no encuentra nada.
    """
    def _norm(name: str) -> str:
        return re.sub(r"\s+", " ", name).strip()

    autores = []

    # 1) Open Library (puede traer múltiples)
    try:
        ol = requests.get(
            "https://openlibrary.org/search.json",
            params={"title": title, "limit": 3},
            timeout=timeout
        )
        ol.raise_for_status()
        docs = ol.json().get("docs", [])
        for d in docs:
            for a in d.get("author_name", []) or []:
                a = _norm(a)
                if a and a not in autores:
                    autores.append(a)
                    if len(autores) >= max_authors:
                        break
            if len(autores) >= max_authors:
                break
    except requests.RequestException as e:
        print(f"[⚠️] OpenLibrary error: {e}")

    # 2) Google Books (fallback, agrega si faltan)
    if len(autores) < max_authors:
        query = f'intitle:"{title}"'
        if category:
            query += f' subject:"{category}"'
        try:
            gb = requests.get(
                "https://www.googleapis.com/books/v1/volumes",
                params={
                    "q": query,
                    "maxResults": 3,
                    "orderBy": "relevance",
                    "langRestrict": lang
                },
                timeout=timeout
            )
            gb.raise_for_status()
            items = gb.json().get("items", []) or []
            for it in items:
                for a in it.get("volumeInfo", {}).get("authors", []) or []:
                    a = _norm(a)
                    if a and a not in autores:
                        autores.append(a)
                        if len(autores) >= max_authors:
                            break
                if len(autores) >= max_authors:
                    break
        except requests.RequestException as e:
            print(f"[⚠️] Google Books error: {e}")

    return autores if autores else ["Desconocido"]


# funcion para obtener el soup de una pagina o una url
def get_soup(url_pagina_soup):
    for intento in range(5):  # 🔹 ADICIÓN: reintentos
        try:
            resp = requests.get(url_pagina_soup, timeout=20)
            resp.raise_for_status()
            return BeautifulSoup(resp.text, "lxml")
        except requests.RequestException:
            print(f"[WARN] Falló {url_pagina_soup} (intento {intento+1}): {e}")
            time.sleep(1.2 * (intento + 1))
    raise RuntimeError(f"No pude obtener {url_pagina_soup}")



# mientras existan paginas siguientes, veridicado por boton next. otra forma de hacerlo es iterando
while next_page:

    soup = get_soup(next_page)
    books = soup.find_all("article", class_="product_pod")

    for b in books:


        # Title
        title = b.h3.a["title"]


        # Price
        # 💥💥💥 
        txtprice = b.find("p", class_="price_color").get_text(strip=True)
        price = float(re.sub(r"[^\d.]", "", txtprice))


        # URL
        href = b.h3.a["href"]
        book_url = urljoin(next_page, href)

        try:
            # detalle 
            soup_detail = get_soup(book_url)
        except RuntimeError as e:
            print(f"[SKIP] No pude abrir detalle: {book_url} -> {e}")
            continue   

        # category
        breadcrumb_link = soup_detail.select("ul.breadcrumb li a")
        category = breadcrumb_link[-1].get_text(strip=True) if len(breadcrumb_link) >= 3 else "Unknown"


        # Rating                   
        rating_tag = soup_detail.select_one("p.star-rating")
        classes = rating_tag.get("class", []) if rating_tag else []                        
        rating_word = next((c for c in classes if c in rating_number), "One")
        rating = rating_number.get(rating_word, 1)                         


        # Stock
        available = soup_detail.find("p", class_="instock availability").text.strip()
        if available == "In stock":
            in_stock = True
        else: 
            in_stock = False


        # description
        description_tag = soup_detail.find("div", id="product_description")
        description = (
            description_tag.find_next_sibling('p').text.strip()
            if description_tag else None
        )


        # 🔹 ADICIÓN (llamar API): buscar autores por título
        # Explicación:
        # - Usamos la función get_autor(title, category).
        # - Retorna una lista [] con los nombres de los autores o Desconocido.
        # - Se agrega como campo "author" en el item para persistirlo luego en JSON.
        authors = get_authors(title, category)
        # Pequeña pausa de cortesía para no saturar la API si hay muchos libros
        time.sleep(0.15)

        items.append({
            "title": title,
            "category": category,
            "rating": rating,
            "URL": book_url,
            "price": price,
            "stock": in_stock,
            "description": description,
            # 🔹 ADICIÓN (nuevo campo en el dataset): autores como lista
            # Explicación:
            # - Este campo nuevo te permitirá luego crear tablas 'autores' y 'libro_autor' (M:N) en tu DB.
            # - Mantenerlo como lista te conserva todos los coautores que reporte la API.
            "authors": authors
        })

    time.sleep(0.5)

    # para cada pagina del 1 al 5
    botton_next = soup.find("li", class_="next")
    if botton_next:
        href_next = botton_next.a["href"]
        next_page = urljoin(next_page, href_next)
        time.sleep(0.15)
    else:
        break


# Guardar en JSON
with open('libros_scrapeados.json', 'w', encoding='utf-8') as f:
    json.dump(items, f, ensure_ascii=False, indent=2)

# Leer desde JSON
with open('libros_scrapeados.json', 'r', encoding='utf-8') as f:
    datos = json.load(f)
    print(f'Se guardaron {len(datos)} libros')
    print(datos[0])  # Mostrar el primero para validar estructura

# Abrir automáticamente el archivo (solo en Windows)
os.startfile('libros_scrapeados.json')


In [12]:
import sqlite3

# 1. Conectamos con la base
# conn representa la conexión a la base de datos.
conn = sqlite3.connect("libros.db")
cursor = conn.cursor()


# 2. Activamos las llaves foráneas (para relaciones entre tablas)
conn.execute("PRAGMA foreign_keys = ON;")   # PRAGMA es una directiva especial de SQLite para configurar opciones




# 3. Escribimos el DDL (definición de tablas)
DDL = """
CREATE TABLE IF NOT EXISTS categories (
    id      INTEGER PRIMARY KEY,
    name    TEXT NOT NULL UNIQUE
);

CREATE TABLE IF NOT EXISTS authors (
    id      INTEGER PRIMARY KEY,
    name    TEXT NOT NULL UNIQUE
);

CREATE TABLE IF NOT EXISTS books (
    id          INTEGER PRIMARY KEY,
    title       TEXT NOT NULL,
    url         TEXT NOT NULL UNIQUE,
    price       REAL NOT NULL,
    rating      INTEGER NOT NULL CHECK (rating BETWEEN 1 AND 5),
    stock       INTEGER NOT NULL CHECK (stock IN (0,1)),
    description TEXT,
    category_id INTEGER NOT NULL,
    FOREIGN KEY (category_id) REFERENCES categories(id) ON DELETE RESTRICT
);

CREATE TABLE IF NOT EXISTS book_author (
    book_id   INTEGER NOT NULL,
    author_id INTEGER NOT NULL,
    PRIMARY KEY (book_id, author_id),
    FOREIGN KEY (book_id)   REFERENCES books(id)   ON DELETE CASCADE,
    FOREIGN KEY (author_id) REFERENCES authors(id) ON DELETE CASCADE
);
"""

# 4. Ejecutamos todas las sentencias
conn.executescript(DDL)

print("✅ Tablas creadas correctamente")

conn.close()


✅ Tablas creadas correctamente


In [4]:
import sqlite3
conn = sqlite3.connect("libros.db")
cursor = conn.cursor()

#Ver que las tablas se crearon
# SELECT sirve para seleccionar datos de una base de datos
# FROM es para especificar la tabla de donde se obtienen los datos
# sqlite_master es una tabla interna que guarda la estructura de la base
# WHERE sirve para filtrar los resultados
# type='table' filtra solo las filas que representan tablas
cursor.execute('''SELECT name FROM sqlite_master WHERE type='table' ''')
# fetchall() obtiene todas las filas del resultado de la consulta
# fetchall() trae una tupla de todas las filas de la tabla o de la base de datos
tablas = cursor.fetchall()

print("Tablas en la base de datos: ")
print(tablas)

conn.close()

Tablas en la base de datos: 
[('categories',), ('authors',), ('books',), ('book_author',)]


In [5]:
import sqlite3, json
from pathlib import Path

DB_PATH = "libros.db"              # cambia si tu DB se llama distinto
JSON_PATH = "libros_scrapeados.json"  # cambia si tu JSON se llama distinto

def cargar_categorias(db_path=DB_PATH, json_path=JSON_PATH):

    # 1. Cargar JSON
    data = json.loads(Path(JSON_PATH).read_text(encoding="utf-8"))
    print(f"Leídos {len(data)} libros desde el JSON ✅ \n")

    # 2. Conexión + FK on
    conn = sqlite3.connect(db_path)
    conn.execute("PRAGMA foreign_keys = ON;")
    cur = conn.cursor()

    # 3. recopilar categorías del JSON
    categorias = { item.get("category") for item in data if item.get("category") }
    print(f"categorías : {categorias}")


    # 4. Insertar en la tabla categories
    cur.executemany( "INSERT OR IGNORE INTO categories(name) VALUES (?);", [(c,) for c in sorted(categorias)])

    # 5. Guardar cambios
    conn.commit()

    # 6. Comprobar el resultado
    total = cur.execute("SELECT COUNT(*) FROM categories;").fetchone()[0]
    print(f"Listo categorías. Total en tabla: {total}")
    conn.close()

# Ejecutar solo este paso primero
cargar_categorias()


Leídos 1000 libros desde el JSON ✅ 

categorías : {'Philosophy', 'Novels', 'Contemporary', 'Health', 'Autobiography', 'New Adult', 'Add a comment', 'Historical Fiction', 'Horror', 'Spirituality', 'Historical', 'Default', 'Nonfiction', 'Science Fiction', 'Crime', 'Adult Fiction', 'Biography', 'Academic', 'Sports and Games', 'Art', 'Cultural', 'Religion', 'Politics', 'Science', 'Self Help', 'Christian', 'Business', 'Christian Fiction', 'Food and Drink', 'Erotica', 'Classics', 'Humor', 'Travel', 'History', 'Music', 'Thriller', 'Fantasy', 'Romance', 'Psychology', 'Poetry', 'Womens Fiction', 'Short Stories', 'Suspense', 'Parenting', 'Young Adult', 'Mystery', 'Sequential Art', 'Paranormal', 'Fiction', 'Childrens'}
Listo categorías. Total en tabla: 50


In [6]:
import sqlite3, json
from pathlib import Path

DB_PATH = "libros.db"
JSON_PATH = "libros_scrapeados.json"

def cargar_authors(db_path=DB_PATH, json_path=JSON_PATH):
    # 1) Leer JSON -> Python
    data = json.loads(Path(json_path).read_text(encoding="utf-8"))

    # 2) Conexión y FK ON
    conn = sqlite3.connect(db_path)
    conn.execute("PRAGMA foreign_keys = ON;")
    cur = conn.cursor()

    # 3) Deduplicar autores (siempre lista)
    autores = set()
    for item in data:
        for nm in item["authors"]:
            autores.add(str(nm).strip())

    # 4) Insert masivo sin duplicar en DB
    cur.executemany(
        "INSERT OR IGNORE INTO authors(name) VALUES (?);",
        [(a,) for a in sorted(autores)]
    )

    # 5) Confirmar y mostrar total
    conn.commit()
    total = cur.execute("SELECT COUNT(*) FROM authors;").fetchone()[0]
    print(f"Listo autores. Total en tabla: {total}")
    conn.close()

# Ejecutar
cargar_authors()


Listo autores. Total en tabla: 1539


### Se inserta datos a la tabla de books


In [ ]:
import sqlite3, json
from pathlib import Path

DB_PATH = "libros.db"
JSON_PATH = "libros_scrapeados.json"



def cargar_books(db_path=DB_PATH, json_path=JSON_PATH):
    data = json.loads(Path(json_path).read_text(encoding="utf-8"))
    conn = sqlite3.connect(db_path)
    conn.execute("PRAGMA foreign_keys = ON;")
    cur = conn.cursor()

    # 1) Obtener mapa categoría nombre->id
    # Precache de categorías name->id para acelerar
    cat_map = dict(cur.execute("SELECT name, id FROM categories;").fetchall())

    # 2) Sentencia de inserción
    insert_sql = """
    INSERT OR IGNORE INTO books(title, url, price, rating, stock, description, category_id)
    VALUES (?, ?, ?, ?, ?, ?, ?);
    """

    # 3) Preparar filas
    filas = []
    for it in data:
        cat = it.get("category")
        cat_id = cat_map.get(cat)

        # Agregar fila, en tupla para la tabla
        # Cada tupla representa una fila a insertar
        filas.append((
            it.get("title"),
            it.get("URL"),
            float(it.get("price") or 0.0),
            int(it.get("rating") or 0),
            int(it.get("stock") or 0),
            it.get("description"),
            cat_id
        ))
        # print(f"Preparando libro: {it.get('title')} (cat_id={cat_id})")

    # 4) Insertar todo
    cur.executemany(insert_sql, filas)
    conn.commit()



    # 5) Confirmar y mostrar total
    total = cur.execute("SELECT COUNT(*) FROM books;").fetchone()[0]
    print(f"Listo books. Total en tabla: {total}")
    conn.close()

# Ejecuta esto como tercer paso
cargar_books()


Listo books. Total en tabla: 1000


### `Se insertan datos a la tabla book_author`
### con relaciones libro ↔ autor

In [1]:
import sqlite3, json
from pathlib import Path

DB_PATH = "libros.db"
JSON_PATH = "libros_scrapeados.json"

def cargar_book_author(db_path=DB_PATH, json_path=JSON_PATH):
    data = json.loads(Path(json_path).read_text(encoding="utf-8"))

    conn = sqlite3.connect(db_path)
    conn.execute("PRAGMA foreign_keys = ON;")
    cur = conn.cursor()


    # 1) obtener mapas libro -> id y autor -> id
    # Cache: url libro -> id
    book_map = dict(cur.execute("SELECT url, id FROM books;").fetchall())
    # Cache: autor -> id
    author_map = dict(cur.execute("SELECT name, id FROM authors;").fetchall())


    # 2) sentencia inserción
    insert_rel = "INSERT OR IGNORE INTO book_author(book_id, author_id) VALUES (?, ?);"



    n_rel = 0
    # 3) recorrer datos del JSON 
    for it in data:
        # 4) obtener book_id por URL
        url = it.get("URL")
        book_id = book_map.get(url)

        # 5) obtener lista de autores
        autores = it.get("authors") or []

        # 6) iterar autores 
        for a in autores:
            # limpiar nombre
            a = str(a).strip()
            # obtener author_id
            a_id = author_map.get(a)

            # insertar relación
            cur.execute(insert_rel, (book_id, a_id))
            n_rel += 1

    # 7) confirmar y mostrar totales
    conn.commit()
    total_rel = cur.execute("SELECT COUNT(*) FROM book_author;").fetchone()[0]
    print(f"Listo relaciones. Agregadas ahora: {n_rel}. Total en tabla: {total_rel}")
    conn.close()

# Ejecuta esto como cuarto paso
cargar_book_author()


Listo relaciones. Agregadas ahora: 2087. Total en tabla: 2087
